<a href="https://colab.research.google.com/github/Starksood/AI-Projects/blob/main/TrainingDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install core Unsloth and missing 2026 kernels
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo liger-kernel
!pip install --no-deps "xformers<0.0.29" "trl<0.12.0" peft accelerate bitsandbytes

# 2. Critical Restart to load kernels
import os
os.kill(os.getpid(), 9)

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-9b8w45ks/unsloth_7ff23237ad7e47fea772996578724d16
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-9b8w45ks/unsloth_7ff23237ad7e47fea772996578724d16
  Resolved https://github.com/unslothai/unsloth.git to commit 53af4a1b3e78f2d1cac9401fe15baf8720fc2303
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.4-py3-none-any.whl size=29571830 sha256=dba59aeca56e9d0ba91e6a9844db6c331c96b8290be4f30649bb9da084928e0f
  Stored in directory: /tmp/pip-ephem-wheel-cache-gj2kwdt7/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.5/276.5 kB 33.8 MB/s eta 0:00:00
   ━━

In [ ]:
import json
from datasets import Dataset

data_path = "memory_finetune_data_v1.jsonl"

examples = []
with open(data_path) as f:
    for line in f:
        obj = json.loads(line)
        convs = obj["conversations"]
        system = next((c["value"] for c in convs if c["from"] == "system"), "")
        human  = next((c["value"] for c in convs if c["from"] == "human"), "")
        gpt    = next((c["value"] for c in convs if c["from"] == "gpt"), "")
        examples.append({"system": system, "human": human, "gpt": gpt})

dataset = Dataset.from_list(examples)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Dataset ready! Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

Dataset ready! Train: 193 | Eval: 22


In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Fix tokenizer (critical)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# 3. Format dataset
def format_example(example):
    text = (
        f"<|im_start|>system\n{example['system']}<|im_end|>\n"
        f"<|im_start|>user\n{example['human']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['gpt']}<|im_end|>"
    )
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=2048,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

train_data = dataset["train"].map(format_example, remove_columns=dataset["train"].column_names)
eval_data  = dataset["test"].map(format_example, remove_columns=dataset["test"].column_names)

# 4. Data collator (simple + stable)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# 5. Training args
training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

# 6. Trainer (NO TRL)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
)

# 7. Train
trainer.train()

# 8. Export GGUF
model.save_pretrained_gguf(
    "model_output",
    tokenizer,
    quantization_method="q4_k_m"
)

print("✅ Training + export complete.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:25<00:00, 102.90s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:13<00:00, 66.56s/it]


Unsloth: Merge process complete. Saved to `/content/model_output`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_output_gguf/qwen2.5-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_output_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model model_output_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to model_output_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f model_output_gguf/Modelfile
✅ Training + export complete.
